In [0]:
# ==============================================================================
# BRONZE LAYER INGESTION: STREAMING VS BATCH INGESTION PIPELINES
# USE CASE: TALENT & RISK ASSET MANAGEMENT
# ==============================================================================

from pyspark.sql.functions import col, current_timestamp, lit

# --- 1. SETUP ENVIRONMENT & STORAGE PATHS ---
spark.sql("CREATE CATALOG IF NOT EXISTS training_medallion")
spark.sql("CREATE SCHEMA IF NOT EXISTS training_medallion.01_bronze")

base_volume = "/Volumes/training_medallion/00_landing/raw_landing_zone"

# Storage paths for Streaming (Incremental audit logs)
stream_landing = f"{base_volume}/streaming_talent_audit"
stream_checkpoint = f"{base_volume}/meta_checkpoints/talent_audit_stream"
stream_schema = f"{base_volume}/meta_schemas/talent_audit_stream"

# Storage paths for Batch (Static master reference table)
batch_landing = f"{base_volume}/batch_risk_parameters"

dbutils.fs.mkdirs(stream_landing)
dbutils.fs.mkdirs(batch_landing)
print("Landing directories created successfully.")


# ==============================================================================
# PIPELINE 1: STREAMING INGESTION (INCREMENTAL AUDIT EVENTS)
# ==============================================================================
# Best suited for continuous, append-only high-velocity events

# Simulate incoming risk transaction event files
stream_raw_data = """{"audit_id": "AUD-901", "employee_id": 1001, "risk_score": 85, "flag_reason": "Excessive System Access"}
{"audit_id": "AUD-902", "employee_id": 1002, "risk_score": 20, "flag_reason": "Routine Check"}"""

dbutils.fs.put(f"{stream_landing}/audit_stream_batch1.json", stream_raw_data, overwrite=True)

# Read streaming source using Auto Loader
df_stream_bronze = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", stream_schema)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .load(stream_landing)
    # Lineage & operational metadata tracking
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("ingest_mode", lit("STREAMING_AUTOLOADER"))
    .withColumn("input_source_file", col("_metadata.file_path"))
)

# Incrementally append new data to Bronze Delta Table using availableNow trigger
query_stream = (
    df_stream_bronze.writeStream
    .format("delta")
    .trigger(availableNow=True) # Runs micro-batches until all available data is consumed, then stops
    .outputMode("append")
    .option("checkpointLocation", stream_checkpoint)
    .option("mergeSchema", "true")
    .table("training_medallion.01_bronze.bronze_talent_risk_events")
)
query_stream.awaitTermination()
print("Pipeline 1 (Streaming Ingestion) completed.")


# ==============================================================================
# PIPELINE 2: BATCH INGESTION (STATIC / SCHEDULED REFERENCE DATA)
# ==============================================================================
# Best suited for periodic snapshots, configuration mappings, or reference dimensions

# Simulate department risk threshold configuration data (CSV)
batch_ref_data = """department_code,department_name,max_risk_threshold,compliance_level
FIN,Finance & Treasury,70,Strict
ENG,Engineering,85,Moderate
HR,Human Resources,60,Strict"""

dbutils.fs.put(f"{batch_landing}/department_risk_matrix.csv", batch_ref_data, overwrite=True)

# Standard Batch Read (Bounded dataset)
df_batch_bronze = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(batch_landing)
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("ingest_mode", lit("BATCH_READ"))
)

# Overwrite or Append to Bronze Delta Table
(
    df_batch_bronze.write
    .format("delta")
    .mode("overwrite") # Overwrite mode is ideal for updating full reference snapshots
    .saveAsTable("training_medallion.01_bronze.bronze_department_risk_parameters")
)
print("Pipeline 2 (Batch Ingestion) completed.")


# ==============================================================================
# VALIDATION & METRIC INSPECTION
# ==============================================================================
print("\n--- 1. Inspecting Streaming Bronze Table (Incremental Audit Logs) ---")
display(spark.table("training_medallion.01_bronze.bronze_talent_risk_events"))

print("\n--- 2. Inspecting Batch Bronze Table (Master Risk Parameters) ---")
display(spark.table("training_medallion.01_bronze.bronze_department_risk_parameters"))

Landing directories created successfully.
Wrote 199 bytes.


26/10/29 14:40:14 Query db291c81-6e67-40f4-86cc-ebbcc74b70bd have exception: org.apache.spark.sql.catalyst.util.UnknownFieldException: [UNKNOWN_FIELD_EXCEPTION.NEW_FIELDS_IN_RECORD_WITH_FILE_PATH] Encountered unknown fields during parsing: {"dept_code":"FIN"}, which can be fixed by an automatic retry: true
 Unknown fields inside file path: /Volumes/training_medallion/00_landing/raw_landing_zone/streaming_talent_audit/audit_events_batch1.json
Rescued record schema: dept_code STRING SQLSTATE: KD003
	at org.apache.spark.sql.catalyst.util.UnknownFieldException.copy(UnknownFieldException.scala:69)
	at org.apache.spark.sql.execution.streaming.runtime.MicroBatchExecution.maybeEvolveSchema(MicroBatchExecution.scala:5425)
	at org.apache.spark.sql.execution.streaming.runtime.MicroBatchExecution.withSchemaEvolution(MicroBatchExecution.scala:5447)
	at org.apache.spark.sql.execution.streaming.runtime.MicroBatchExecution.$anonfun$executeOneBatch$4(MicroBatchExecution.scala:2622)
	at scala.runtime.ja

---------------------------------------------------------------------------
StreamingQueryException                   Traceback (most recent call last)
File <command-5685452990588885>, line 63
     53 # Incrementally append new data to Bronze Delta Table using availableNow trigger
     54 query_stream = (
     55     df_stream_bronze.writeStream
     56     .format("delta")
   (...)     61     .table("training_medallion.01_bronze.bronze_talent_risk_events")
     62 )
---> 63 query_stream.awaitTermination()
     64 print("Pipeline 1 (Streaming Ingestion) completed.")
     67 # ==============================================================================
     68 # PIPELINE 2: BATCH INGESTION (STATIC / SCHEDULED REFERENCE DATA)
     69 # ==============================================================================
     70 # Best suited for periodic snapshots, configuration mappings, or reference dimensions
     71 
     72 # Simulate department risk threshold configuration data (CSV)

F